In [1]:
!pip install transformers datasets gradio torch scikit-learn rouge-score sentencepiece sentence-transformers -q

  Preparing metadata (setup.py) ... done


In [2]:
import torch
import gradio as gr
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from rouge_score import rouge_scorer

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
from datasets import load_dataset

dataset = load_dataset("kdave/Indian_Financial_News")
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

training_data_26000.csv:   0%|          | 0.00/115M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26961 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['URL', 'Content', 'Summary', 'Sentiment'],
        num_rows: 26961
    })
})
{'URL': 'https://www.moneycontrol.com/news/business/economy/covid-19-pandemic-us-consumer-spending-tumble-in-april-5334861.html', 'Content': 'US consumer spending dropped by a record in April as the COVID-19 pandemic undercut demand, buttressing expectations that the economy could contract in the second quarter at its steepest pace since the Great Depression.\n\nThe Commerce Department said on Friday consumer spending, which accounts for more than two-thirds of U.S. economic activity, plunged 13.6 percent last month. That was the biggest drop since the government started tracking series in 1959, and followed a 6.9 percent tumble in March.\n\nEconomists polled by Reuters had forecast consumer spending plummeting 12.6 percent in April.\n\nFollow our full coverage of the coronavirus pandemic here.', 'Summary': 'consumer spending plunges 13.6 percent in April. 

In [5]:
sample = dataset["train"][0]
print("Keys available:", sample.keys())
for key, value in sample.items():
    print(f"\n{key}:")
    print(str(value)[:300])

Keys available: dict_keys(['URL', 'Content', 'Summary', 'Sentiment'])

URL:
https://www.moneycontrol.com/news/business/economy/covid-19-pandemic-us-consumer-spending-tumble-in-april-5334861.html

Content:
US consumer spending dropped by a record in April as the COVID-19 pandemic undercut demand, buttressing expectations that the economy could contract in the second quarter at its steepest pace since the Great Depression.

The Commerce Department said on Friday consumer spending, which accounts for mo

Summary:
consumer spending plunges 13.6 percent in April. that was the biggest drop since the government started tracking series in 1959. consumer spending accounts for more than two-thirds of economic activity. economists polled by Reuters had forecast consumer spending plummeting 12.6 percent. a spokesman 

Sentiment:
Negative


In [6]:
def prepare_data(examples):
    # Use Content as article and Summary as target
    articles = examples["Content"]
    summaries = examples["Summary"]
    return {"article": articles, "summary": summaries}

dataset = dataset.map(prepare_data, batched=True, remove_columns=dataset["train"].column_names)
print(dataset)
print(dataset["train"][0])

Map:   0%|          | 0/26961 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['article', 'summary'],
        num_rows: 26961
    })
})
{'article': 'US consumer spending dropped by a record in April as the COVID-19 pandemic undercut demand, buttressing expectations that the economy could contract in the second quarter at its steepest pace since the Great Depression.\n\nThe Commerce Department said on Friday consumer spending, which accounts for more than two-thirds of U.S. economic activity, plunged 13.6 percent last month. That was the biggest drop since the government started tracking series in 1959, and followed a 6.9 percent tumble in March.\n\nEconomists polled by Reuters had forecast consumer spending plummeting 12.6 percent in April.\n\nFollow our full coverage of the coronavirus pandemic here.', 'summary': 'consumer spending plunges 13.6 percent in April. that was the biggest drop since the government started tracking series in 1959. consumer spending accounts for more than two-thirds of economic activ

In [7]:
# Split train into train and temp
split1 = dataset["train"].train_test_split(test_size=0.2, seed=42)
# Split temp into validation and test
split2 = split1["test"].train_test_split(test_size=0.5, seed=42)

from datasets import DatasetDict
dataset = DatasetDict({
    "train": split1["train"],
    "validation": split2["train"],
    "test": split2["test"]
})

print(f"Train: {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test: {len(dataset['test'])}")

Train: 21568
Validation: 2696
Test: 2697


In [8]:
model_checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
print("Tokenizer loaded!")

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded!


In [9]:
max_input_length = 512
max_target_length = 128

def preprocess(examples):
    inputs = tokenizer(
        examples["article"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )
    targets = tokenizer(
        text_target=examples["summary"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized_dataset = dataset.map(
    preprocess,
    batched=True,
    remove_columns=dataset["train"].column_names
)
print("Tokenization done!")
print(tokenized_dataset)

Map:   0%|          | 0/21568 [00:00<?, ? examples/s]

Map:   0%|          | 0/2696 [00:00<?, ? examples/s]

Map:   0%|          | 0/2697 [00:00<?, ? examples/s]

Tokenization done!
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 21568
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2696
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2697
    })
})


In [10]:
small_train = tokenized_dataset["train"].select(range(1000))
small_val = tokenized_dataset["validation"].select(range(200))
small_test = tokenized_dataset["test"].select(range(200))

print(f"Train: {len(small_train)}")
print(f"Validation: {len(small_val)}")
print(f"Test: {len(small_test)}")

Train: 1000
Validation: 200
Test: 200


In [11]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
model.to(device)
print("BART model loaded!")

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

BART model loaded!


In [12]:
import numpy as np
from rouge_score import rouge_scorer as rs

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scorer = rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    r1 = r2 = rl = 0
    for pred, label in zip(decoded_preds, decoded_labels):
        scores = scorer.score(label, pred)
        r1 += scores["rouge1"].fmeasure
        r2 += scores["rouge2"].fmeasure
        rl += scores["rougeL"].fmeasure

    n = len(decoded_preds)
    return {
        "rouge1": round(r1/n, 4),
        "rouge2": round(r2/n, 4),
        "rougeL": round(rl/n, 4)
    }

print("Metrics function ready!")

Metrics function ready!


In [13]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./fns_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    predict_with_generate=True,
    fp16=torch.cuda.is_available()
)
print("Training arguments set!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training arguments set!


In [14]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
print("Trainer ready!")

Trainer ready!


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel
1,1.573312,1.147147,0.245700,0.163000,0.222100
2,0.765387,0.582199,0.258300,0.181600,0.237600
3,0.658054,0.534635,0.256700,0.177800,0.235800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=375, training_loss=1.5441183433532715, metrics={'train_runtime': 181.3008, 'train_samples_per_second': 16.547, 'train_steps_per_second': 2.068, 'total_flos': 914604687360000.0, 'train_loss': 1.5441183433532715, 'epoch': 3.0})

In [16]:
results = trainer.evaluate(small_test)
print("\nTest Results:")
for key, value in results.items():
    print(f"{key}: {round(value, 4)}")


Test Results:
eval_loss: 0.5288
eval_rouge1: 0.2486
eval_rouge2: 0.167
eval_rougeL: 0.2223
eval_runtime: 14.1332
eval_samples_per_second: 14.151
eval_steps_per_second: 1.769
epoch: 3.0


In [17]:
model.save_pretrained("./fns_model_final")
tokenizer.save_pretrained("./fns_model_final")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!


In [18]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

def extractive_summary(text, num_sentences=3):
    sentences = [s.strip() for s in text.split(".") if len(s.strip()) > 20]
    if len(sentences) <= num_sentences:
        return ". ".join(sentences) + "."

    embeddings = sentence_model.encode(sentences)
    doc_embedding = np.mean(embeddings, axis=0).reshape(1, -1)
    similarities = cosine_similarity(doc_embedding, embeddings)[0]

    top_indices = sorted(np.argsort(similarities)[-num_sentences:])
    return ". ".join([sentences[i] for i in top_indices]) + "."

print("Extractive summarization ready!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Extractive summarization ready!


In [19]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

sum_tokenizer = AutoTokenizer.from_pretrained("./fns_model_final")
sum_model = AutoModelForSeq2SeqLM.from_pretrained("./fns_model_final")
sum_model.to(device)

def abstractive_summary(text):
    inputs = sum_tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = sum_model.generate(
        inputs["input_ids"],
        max_length=130,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    summary = sum_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

print("Abstractive summarization ready!")

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Abstractive summarization ready!


In [20]:
def analyze_financial_news(article_text, method):
    if not article_text.strip():
        return "⚠️ Please paste a financial news article!"

    if method == "Extractive (Sentence Ranking via Semantic Similarity)":
        summary = extractive_summary(article_text, num_sentences=3)
        method_note = "Extractive — Top sentences ranked by semantic similarity (like Bloomberg)"
    else:
        summary = abstractive_summary(article_text)
        method_note = "Abstractive — BART fine-tuned on financial news corpus"

    output = f"""
📰 FINANCIAL NEWS SUMMARIZER
==============================

📌 METHOD USED
--------------
{method_note}

📝 GENERATED SUMMARY
--------------
{summary}

==============================
✅ Summarization Complete
"""
    return output


demo = gr.Interface(
    fn=analyze_financial_news,
    inputs=[
        gr.Textbox(
            lines=15,
            placeholder="Paste a financial news article here...",
            label="Financial News Article"
        ),
        gr.Radio(
            choices=[
                "Extractive (Sentence Ranking via Semantic Similarity)",
                "Abstractive (BART fine-tuned on Financial Corpus)"
            ],
            value="Extractive (Sentence Ranking via Semantic Similarity)",
            label="Summarization Method"
        )
    ],
    outputs=gr.Textbox(label="📊 Summary Report", lines=15),
    title="📈 Financial News Summarizer",
    description="Paste any financial news article and get an instant briefing using Extractive or Abstractive methods — inspired by Bloomberg's NLP pipeline"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://11c249b0482d66130f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
